# 1\.4\.3 Evaluate Retrospective Clustering Representations

This notebook determines how Taxi Zones should be represented before retrospective mobility\-neighborhood clustering\. The analytical unit is Taxi Zone × pre\_post\_cp, and the goal is to define a small, defensible set of clustering inputs that preserves both overall mobility level and temporal shape without introducing unnecessary redundancy or complexity\.

The analysis begins with seven baseline measures: Taxi trip count and average speed, FHVHV trip count and average speed, Subway ridership, Bus trip count, and average Bus speed\. It then evaluates Taxi and FHVHV trip distance and duration, together with Subway Transfers, as supplemental candidates\. The notebook does not select the final clustering algorithm or number of clusters; it ends with a clear written specification of the matrix or small set of matrices that Notebook 3\.1\.1 should construct\.

In [1]:
# ---------------------------------------------------------------------
# 1.4.3 Setup: imports and display settings
# ---------------------------------------------------------------------

from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

from project_branding import (
    BRAND_COLORS,
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)

print("1.4.3 setup imports loaded.")

1.4.3 setup imports loaded.


In [2]:
# ---------------------------------------------------------------------
# 1.4.3 Setup: paths, toggles, and representation constants
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")
MERGED_FINAL_DIR = PIPELINE_DATA_DIR / "1.3.1.final_tables"
REPRESENTATION_OUTPUT_DIR = PIPELINE_DATA_DIR / "1.4.3.representation_review"

ANALYSIS_PANEL_PATH = MERGED_FINAL_DIR / "analysis_ready_mobility_panel.parquet"
TAXI_ZONE_CONTEXT_PATH = MERGED_FINAL_DIR / "nyc_taxi_zones_harmonized.parquet"

REPRESENTATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STUDY_START_DATE = pd.Timestamp("2023-01-01")
STUDY_END_DATE = pd.Timestamp("2026-03-31")
CONGESTION_PRICING_START_DATE = pd.Timestamp("2025-01-05")

RUN_REPRESENTATION_QA = True
SAVE_REPRESENTATION_OUTPUTS = True

BASELINE_REPRESENTATION_METRICS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
    "bus_trip_count",
    "avg_bus_speed",
]

SUPPLEMENTAL_CANDIDATE_METRICS = [
    "taxi_avg_trip_distance",
    "taxi_avg_trip_duration",
    "fhvhv_avg_trip_distance",
    "fhvhv_avg_trip_duration",
    "subway_transfers",
]

PRIMARY_CLUSTERING_METRICS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
    "bus_trip_count",
    "avg_bus_speed",
    "taxi_avg_trip_duration",
    "fhvhv_avg_trip_duration",
    "subway_transfers",
]

SENSITIVITY_CANDIDATE_METRICS = [
    "taxi_avg_trip_distance",
    "fhvhv_avg_trip_distance",
]

REPRESENTATION_REVIEW_METRICS = sorted(
    set(
        BASELINE_REPRESENTATION_METRICS
        + SUPPLEMENTAL_CANDIDATE_METRICS
    )
)

COUNT_LIKE_METRICS = [
    "taxi_trip_count",
    "fhvhv_trip_count",
    "subway_ridership",
    "bus_trip_count",
    "subway_transfers",
]

AVERAGE_LIKE_METRICS = [
    "taxi_avg_trip_speed",
    "fhvhv_avg_trip_speed",
    "avg_bus_speed",
    "taxi_avg_trip_distance",
    "taxi_avg_trip_duration",
    "fhvhv_avg_trip_distance",
    "fhvhv_avg_trip_duration",
]

METRIC_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi average speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV average speed",
    "subway_ridership": "Subway ridership",
    "bus_trip_count": "Bus trips",
    "avg_bus_speed": "Average bus speed",
    "taxi_avg_trip_distance": "Taxi average distance",
    "taxi_avg_trip_duration": "Taxi average duration",
    "fhvhv_avg_trip_distance": "FHVHV average distance",
    "fhvhv_avg_trip_duration": "FHVHV average duration",
    "subway_transfers": "Subway transfers",
}

TEMPORAL_BUCKET_ORDER = [
    "weekday_overnight",
    "weekday_am_peak",
    "weekday_midday",
    "weekday_pm_peak",
    "weekday_evening",
    "weekend_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_pm_peak",
    "weekend_evening",
]

METADATA_COLUMNS = [
    "taxi_zone_id",
    "zone",
    "borough",
    "canonical_location_id",
    "cbd_spatial_category",
    "pre_post_cp",
]

GRAIN_COLUMNS = ["taxi_zone_id", "pre_post_cp"]
TEMPORAL_GRAIN_COLUMNS = ["taxi_zone_id", "pre_post_cp", "temporal_bucket"]

setup_summary_df = pd.DataFrame(
    {
        "setting": [
            "analysis_panel_path",
            "representation_output_dir",
            "baseline_metric_count",
            "supplemental_candidate_count",
            "primary_clustering_metric_count",
            "sensitivity_candidate_count",
            "save_representation_outputs",
        ],
        "value": [
            str(ANALYSIS_PANEL_PATH),
            str(REPRESENTATION_OUTPUT_DIR),
            len(BASELINE_REPRESENTATION_METRICS),
            len(SUPPLEMENTAL_CANDIDATE_METRICS),
            len(PRIMARY_CLUSTERING_METRICS),
            len(SENSITIVITY_CANDIDATE_METRICS),
            SAVE_REPRESENTATION_OUTPUTS,
        ],
    }
)

display(setup_summary_df)

,setting,value
0,analysis_panel_path,pipeline_data/1.3.1.final_tables/analysis_read...
1,representation_output_dir,pipeline_data/1.4.3.representation_review
2,baseline_metric_count,7
3,supplemental_candidate_count,5
4,primary_clustering_metric_count,10
5,sensitivity_candidate_count,2
6,save_representation_outputs,True


In [3]:
# ---------------------------------------------------------------------
# 1.4.3 Setup: lightweight helpers
# ---------------------------------------------------------------------

def sql_path(path: Path) -> str:
    """Return a DuckDB-safe path string."""
    return path.as_posix().replace("'", "''")


def require_columns(available_columns, required_columns, table_name):
    """Raise a clear error if a required source column is unavailable."""
    missing_columns = sorted(set(required_columns) - set(available_columns))
    if missing_columns:
        raise AssertionError(
            f"{table_name} is missing required columns: {missing_columns}"
        )


def zscore_series(series: pd.Series) -> pd.Series:
    """Standardize a numeric Series while preserving nulls and avoiding divide-by-zero."""
    numeric_series = pd.to_numeric(series, errors="coerce")
    std_value = numeric_series.std(ddof=0)
    if pd.isna(std_value) or std_value == 0:
        return pd.Series(np.nan, index=series.index)
    return (numeric_series - numeric_series.mean()) / std_value


def weighted_average(value_series: pd.Series, weight_series: pd.Series) -> float:
    """Compute a weighted average for diagnostics that compare average-like metrics."""
    valid_mask = value_series.notna() & weight_series.notna() & (weight_series > 0)
    if not valid_mask.any():
        return np.nan
    return np.average(value_series.loc[valid_mask], weights=weight_series.loc[valid_mask])

print("1.4.3 setup helpers loaded.")

1.4.3 setup helpers loaded.


In [4]:
# ---------------------------------------------------------------------
# 1.4.3 Setup: load 1.3.1 analysis panel
# ---------------------------------------------------------------------

analysis_panel_schema_df = duckdb.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{sql_path(ANALYSIS_PANEL_PATH)}')
""").df()

analysis_panel_columns = analysis_panel_schema_df["column_name"].tolist()

REQUIRED_ANALYSIS_PANEL_COLUMNS = [
    "taxi_zone_id",
    "zone",
    "borough",
    "canonical_location_id",
    "cbd_spatial_category",
    "date",
    "pre_post_cp",
    "temporal_bucket",
    "temporal_bucket_order",
    *REPRESENTATION_REVIEW_METRICS,
]

require_columns(
    analysis_panel_columns,
    REQUIRED_ANALYSIS_PANEL_COLUMNS,
    "1.3.1 analysis-ready mobility panel",
)

mobility_panel_df = pd.read_parquet(
    ANALYSIS_PANEL_PATH,
    columns=REQUIRED_ANALYSIS_PANEL_COLUMNS,
)

mobility_panel_df["date"] = pd.to_datetime(mobility_panel_df["date"])
mobility_panel_df["temporal_bucket"] = pd.Categorical(
    mobility_panel_df["temporal_bucket"],
    categories=TEMPORAL_BUCKET_ORDER,
    ordered=True,
)

analysis_panel_load_summary_df = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "taxi_zone_count",
            "policy_period_count",
            "temporal_bucket_count",
            "min_date",
            "max_date",
            "duplicate_zone_date_bucket_rows",
        ],
        "value": [
            len(mobility_panel_df),
            mobility_panel_df["taxi_zone_id"].nunique(),
            mobility_panel_df["pre_post_cp"].nunique(),
            mobility_panel_df["temporal_bucket"].nunique(),
            mobility_panel_df["date"].min(),
            mobility_panel_df["date"].max(),
            mobility_panel_df.duplicated(["taxi_zone_id", "date", "temporal_bucket"]).sum(),
        ],
    }
)

display(analysis_panel_load_summary_df)

,metric,value
0,row_count,1559590
1,taxi_zone_count,263
2,policy_period_count,2
3,temporal_bucket_count,10
4,min_date,2023-01-01 00:00:00
5,max_date,2026-03-31 00:00:00
6,duplicate_zone_date_bucket_rows,0


## 1\.4\.3\.1 Supplemental Representation Choices

This section asks whether the supplemental measures add useful Taxi Zone identity at the same grain we expect clustering to use: \`taxi\_zone\_id × pre\_post\_cp\`\. That is different from the earlier 1\.4\.1 shortlist work, which focused on recurring EDA visuals\. Here we care about whether distance, duration, and transfers identify different zones after the baseline mobility signals are already in the representation\.

The analysis keeps the question practical\. We first collapse the panel to a zone\-period review frame, then test supplemental metrics through same\-grain correlations, residual signal beyond baseline metrics, top\-zone overlap, and pre/post stability\. Subway transfers get a focused transfer\-hub check because that metric may be useful even if it overlaps with ridership in broad citywide summaries\.

### Build Taxi Zone\-Period Review Frame

First we collapse the analysis panel to the representation\-review grain\. This is not the final clustering matrix yet; it is a compact review frame that lets us evaluate whether supplemental metrics add stable zone\-period identity beyond the baseline metrics\.

In [5]:
# ---------------------------------------------------------------------
# Build Taxi Zone-period representation review frame
# ---------------------------------------------------------------------

# Preserve one descriptive value for each metadata field within every
# Taxi Zone × policy-period group. These fields should be constant within
# the grouping grain, so "first" simply carries them into the summary row.
zone_period_aggregation_rules = {
    "zone": "first",
    "borough": "first",
    "canonical_location_id": "first",
    "cbd_spatial_category": "first",
}

# Summarize each baseline and supplemental review metric using its
# Taxi Zone-period mean. Mean and median will be compared later; this
# initial frame provides a consistent period-level starting point.
zone_period_aggregation_rules.update(
    {
        metric: "mean"
        for metric in REPRESENTATION_REVIEW_METRICS
    }
)

# Collapse the mobility panel to one row per Taxi Zone × pre/post period.
zone_period_representation_df = (
    mobility_panel_df
    .groupby(
        GRAIN_COLUMNS,
        observed=True,
    )
    .agg(zone_period_aggregation_rules)
    .reset_index()
)

# Keep only the approved metadata fields and the metrics currently under
# representation review. This prevents unrelated panel columns from
# entering the analysis accidentally.
zone_period_representation_df = (
    zone_period_representation_df[
        METADATA_COLUMNS
        + REPRESENTATION_REVIEW_METRICS
    ]
    .copy()
)

# Create a compact QA summary confirming the resulting grain and the size
# of the baseline and supplemental metric sets under consideration.
zone_period_review_summary_df = pd.DataFrame(
    {
        "metric": [
            "zone_period_rows",
            "taxi_zone_count",
            "policy_period_count",
            "baseline_metric_count",
            "supplemental_candidate_count",
        ],
        "value": [
            len(zone_period_representation_df),
            zone_period_representation_df[
                "taxi_zone_id"
            ].nunique(),
            zone_period_representation_df[
                "pre_post_cp"
            ].nunique(),
            len(BASELINE_REPRESENTATION_METRICS),
            len(SUPPLEMENTAL_CANDIDATE_METRICS),
        ],
    }
)

display(zone_period_review_summary_df)

,metric,value
0,zone_period_rows,526
1,taxi_zone_count,263
2,policy_period_count,2
3,baseline_metric_count,7
4,supplemental_candidate_count,5


Findings\. The representation review frame contains the expected 526 Taxi Zone\-period profiles, corresponding to all 263 Taxi Zones observed once in each of the two congestion\-pricing periods\. The review includes the seven established baseline mobility measures and five supplemental candidates, confirming that the analysis is operating at the intended \`Taxi Zone × pre\_post\_cp\` grain with the full 12\-measure consideration set available for representation testing\.

### Review Supplemental Overlap at the Clustering Grain

Now we rerun the overlap question at the actual zone\-period review grain\. High correlation here is not automatically bad, but it tells us when a supplemental metric may mostly be repeating a baseline signal\.

In [6]:
# ---------------------------------------------------------------------
# Review supplemental overlap at the clustering grain
# ---------------------------------------------------------------------

# Define the baseline or related measures against which each supplemental
# candidate should be evaluated. The comparisons focus on whether the
# candidate adds distinct Taxi Zone-period information or largely repeats
# an existing demand, speed, or trip-characteristic signal.
SUPPLEMENTAL_COMPARISON_METRICS = {
    "taxi_avg_trip_distance": [
        "taxi_trip_count",
        "taxi_avg_trip_speed",
        "taxi_avg_trip_duration",
    ],
    "taxi_avg_trip_duration": [
        "taxi_trip_count",
        "taxi_avg_trip_speed",
        "taxi_avg_trip_distance",
    ],
    "fhvhv_avg_trip_distance": [
        "fhvhv_trip_count",
        "fhvhv_avg_trip_speed",
        "fhvhv_avg_trip_duration",
    ],
    "fhvhv_avg_trip_duration": [
        "fhvhv_trip_count",
        "fhvhv_avg_trip_speed",
        "fhvhv_avg_trip_distance",
    ],
    "subway_transfers": [
        "subway_ridership",
    ],
}

# Collect one record for every candidate-versus-comparison pairing.
supplemental_overlap_records = []

for candidate_metric, comparison_metrics in (
    SUPPLEMENTAL_COMPARISON_METRICS.items()
):
    for comparison_metric in comparison_metrics:

        # Restrict each correlation to Taxi Zone-period rows where both
        # measures are observed so the matched sample is explicit.
        comparison_df = (
            zone_period_representation_df[
                [
                    candidate_metric,
                    comparison_metric,
                ]
            ]
            .dropna()
        )

        # Correlation is undefined when either measure is constant at this
        # grain, so require variation in both columns before calculating it.
        has_variation = (
            comparison_df[candidate_metric].nunique() > 1
            and comparison_df[comparison_metric].nunique() > 1
        )

        # Use Spearman correlation because the goal is to assess overlap in
        # Taxi Zone-period ordering without assuming a linear relationship.
        # Require at least ten matched observations for a minimally stable
        # comparison.
        spearman_correlation = (
            comparison_df[candidate_metric].corr(
                comparison_df[comparison_metric],
                method="spearman",
            )
            if len(comparison_df) >= 10 and has_variation
            else np.nan
        )

        # Store both the signed and absolute correlations. The signed value
        # preserves direction, while the absolute value supports redundancy
        # screening regardless of whether the relationship is positive or
        # negative.
        supplemental_overlap_records.append(
            {
                "candidate_metric": candidate_metric,
                "candidate_label": METRIC_LABELS[
                    candidate_metric
                ],
                "comparison_metric": comparison_metric,
                "comparison_label": METRIC_LABELS[
                    comparison_metric
                ],
                "spearman_correlation": (
                    round(spearman_correlation, 3)
                    if pd.notna(spearman_correlation)
                    else np.nan
                ),
                "abs_spearman_correlation": (
                    round(abs(spearman_correlation), 3)
                    if pd.notna(spearman_correlation)
                    else np.nan
                ),
                "matched_zone_periods": len(comparison_df),
            }
        )

# Convert the comparison records into a compact review table and sort each
# candidate's comparisons from strongest to weakest absolute association.
supplemental_same_grain_overlap_df = (
    pd.DataFrame(supplemental_overlap_records)
    .sort_values(
        [
            "candidate_metric",
            "abs_spearman_correlation",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(supplemental_same_grain_overlap_df)

,candidate_metric,candidate_label,comparison_metric,comparison_label,spearman_correlation,abs_spearman_correlation,matched_zone_periods
0,fhvhv_avg_trip_distance,FHVHV average distance,fhvhv_avg_trip_speed,FHVHV average speed,0.691,0.691,525
1,fhvhv_avg_trip_distance,FHVHV average distance,fhvhv_avg_trip_duration,FHVHV average duration,0.528,0.528,525
2,fhvhv_avg_trip_distance,FHVHV average distance,fhvhv_trip_count,FHVHV trips,-0.291,0.291,525
3,fhvhv_avg_trip_duration,FHVHV average duration,fhvhv_avg_trip_distance,FHVHV average distance,0.528,0.528,525
4,fhvhv_avg_trip_duration,FHVHV average duration,fhvhv_trip_count,FHVHV trips,0.257,0.257,525
5,fhvhv_avg_trip_duration,FHVHV average duration,fhvhv_avg_trip_speed,FHVHV average speed,-0.194,0.194,525
6,subway_transfers,Subway transfers,subway_ridership,Subway ridership,0.619,0.619,309
7,taxi_avg_trip_distance,Taxi average distance,taxi_avg_trip_duration,Taxi average duration,0.833,0.833,525
8,taxi_avg_trip_distance,Taxi average distance,taxi_trip_count,Taxi trips,-0.750,0.750,525
9,taxi_avg_trip_distance,Taxi average distance,taxi_avg_trip_speed,Taxi average speed,0.719,0.719,525


Findings\. The supplemental measures vary substantially in how much they overlap with the established baseline\. Taxi average distance and duration are highly related to one another \(Spearman ρ = 0\.833\) and are also strongly associated with Taxi trip count and average speed, suggesting that retaining both would likely add redundancy\. FHVHV distance and duration are less interchangeable: distance is moderately related to speed \(ρ = 0\.691\) and duration \(ρ = 0\.528\), while duration has only weak relationships with FHVHV trip count and speed, indicating that it may contribute a more distinct trip\-composition signal\. Subway Transfers is moderately associated with Subway Ridership \(ρ = 0\.619\) across the 309 matched Taxi Zone\-period profiles, supporting the earlier conclusion that Transfers captures related but non\-equivalent information\. Overall, the results support a selective rather than automatic expansion of the seven\-metric baseline\.

### Measure Residual Zone Signal Beyond the Baseline

This is the key check\. For each supplemental candidate, we ask how much signal remains after accounting for the baseline metric or metrics that should already explain it\. If a candidate still has meaningful residual spread, it may add a different kind of zone identity\.

In [7]:
# ---------------------------------------------------------------------
# Measure residual zone signal beyond the baseline
# ---------------------------------------------------------------------

# Define the established baseline predictors for each supplemental candidate.
# The goal is to estimate how much Taxi Zone-period variation remains in the
# candidate after accounting for the related demand and speed measures already
# present in the seven-metric baseline.
SUPPLEMENTAL_BASELINE_PREDICTORS = {
    "taxi_avg_trip_distance": [
        "taxi_trip_count",
        "taxi_avg_trip_speed",
    ],
    "taxi_avg_trip_duration": [
        "taxi_trip_count",
        "taxi_avg_trip_speed",
    ],
    "fhvhv_avg_trip_distance": [
        "fhvhv_trip_count",
        "fhvhv_avg_trip_speed",
    ],
    "fhvhv_avg_trip_duration": [
        "fhvhv_trip_count",
        "fhvhv_avg_trip_speed",
    ],
    "subway_transfers": [
        "subway_ridership",
    ],
}

# Store one summary record per supplemental candidate and retain the
# Taxi Zone-period residual values for later inspection of unusual zones.
supplemental_residual_records = []
supplemental_residual_value_frames = []

for candidate_metric, predictor_metrics in (
    SUPPLEMENTAL_BASELINE_PREDICTORS.items()
):
    # Restrict each model to the candidate, its baseline predictors, and the
    # metadata needed to interpret the resulting residuals.
    model_columns = [candidate_metric] + predictor_metrics

    model_df = (
        zone_period_representation_df[
            METADATA_COLUMNS + model_columns
        ]
        .dropna(subset=model_columns)
        .copy()
    )

    # Standardize the candidate and predictors so the regression evaluates
    # relative Taxi Zone-period patterns rather than differences in units.
    y = zscore_series(
        model_df[candidate_metric]
    )

    x_df = model_df[
        predictor_metrics
    ].apply(zscore_series)

    # Remove rows that become undefined during standardization, such as cases
    # where a metric has no usable variation in the matched sample.
    valid_mask = (
        y.notna()
        & x_df.notna().all(axis=1)
    )

    model_df = model_df.loc[
        valid_mask
    ].copy()

    y = y.loc[
        valid_mask
    ]

    x_df = x_df.loc[
        valid_mask
    ]

    # Require a minimally adequate sample and variation in the candidate.
    # Otherwise, return undefined model and residual diagnostics.
    if (
        len(model_df) < 10
        or y.nunique() <= 1
    ):
        baseline_r2 = np.nan

        residual_values = pd.Series(
            np.nan,
            index=model_df.index,
        )

    else:
        # Fit a simple ordinary least-squares model with an intercept.
        # The fitted values represent the candidate pattern explained by the
        # established baseline measures; the residuals capture what remains.
        design_matrix = np.column_stack(
            [
                np.ones(len(x_df)),
                x_df.to_numpy(),
            ]
        )

        coefficients = np.linalg.lstsq(
            design_matrix,
            y.to_numpy(),
            rcond=None,
        )[0]

        fitted_values = (
            design_matrix
            @ coefficients
        )

        residual_values = pd.Series(
            y.to_numpy() - fitted_values,
            index=model_df.index,
        )

        # Calculate R² to quantify the share of candidate variation explained
        # by the baseline predictors at the Taxi Zone-period grain.
        total_sum_squares = np.square(
            y.to_numpy() - y.mean()
        ).sum()

        residual_sum_squares = np.square(
            residual_values.to_numpy()
        ).sum()

        baseline_r2 = (
            1
            - (
                residual_sum_squares
                / total_sum_squares
            )
            if total_sum_squares > 0
            else np.nan
        )

    # Summarize the remaining unexplained signal. Larger residual spread means
    # the candidate retains more Taxi Zone-period variation beyond the baseline.
    residual_iqr = (
        residual_values.quantile(0.75)
        - residual_values.quantile(0.25)
    )

    supplemental_residual_records.append(
        {
            "candidate_metric": candidate_metric,
            "candidate_label": METRIC_LABELS[
                candidate_metric
            ],
            "baseline_predictors": ", ".join(
                METRIC_LABELS[metric]
                for metric in predictor_metrics
            ),
            "valid_zone_periods": len(model_df),
            "baseline_r2": (
                round(baseline_r2, 3)
                if pd.notna(baseline_r2)
                else np.nan
            ),
            "residual_std": (
                round(
                    residual_values.std(ddof=0),
                    3,
                )
                if residual_values.notna().any()
                else np.nan
            ),
            "residual_iqr": (
                round(residual_iqr, 3)
                if pd.notna(residual_iqr)
                else np.nan
            ),
            "residual_abs_p90": (
                round(
                    residual_values
                    .abs()
                    .quantile(0.90),
                    3,
                )
                if residual_values.notna().any()
                else np.nan
            ),
        }
    )

    # Preserve the residual for every matched Taxi Zone-period so later cells
    # can identify where the candidate differs most from its baseline pattern.
    residual_frame = model_df[
        METADATA_COLUMNS
    ].copy()

    residual_frame[
        "candidate_metric"
    ] = candidate_metric

    residual_frame[
        "candidate_label"
    ] = METRIC_LABELS[
        candidate_metric
    ]

    residual_frame[
        "candidate_residual_z"
    ] = residual_values

    supplemental_residual_value_frames.append(
        residual_frame
    )

# Assemble the candidate-level diagnostic table and the long-form residual
# table used for later Taxi Zone-period interpretation.
supplemental_residual_signal_df = pd.DataFrame(
    supplemental_residual_records
)

supplemental_residual_values_df = pd.concat(
    supplemental_residual_value_frames,
    ignore_index=True,
)

display(
    supplemental_residual_signal_df
)

,candidate_metric,candidate_label,baseline_predictors,valid_zone_periods,baseline_r2,residual_std,residual_iqr,residual_abs_p90
0,taxi_avg_trip_distance,Taxi average distance,"Taxi trips, Taxi average speed",525,0.603,0.630,0.740,0.858
1,taxi_avg_trip_duration,Taxi average duration,"Taxi trips, Taxi average speed",525,0.300,0.837,1.086,1.202
2,fhvhv_avg_trip_distance,FHVHV average distance,"FHVHV trips, FHVHV average speed",525,0.547,0.673,0.655,0.824
3,fhvhv_avg_trip_duration,FHVHV average duration,"FHVHV trips, FHVHV average speed",525,0.196,0.897,1.023,1.191
4,subway_transfers,Subway transfers,Subway ridership,309,0.152,0.921,0.375,0.813


Findings\. The residual analysis shows that all five supplemental candidates retain some Taxi Zone\-period signal beyond their related baseline measures, but the amount varies considerably\. Taxi average distance is substantially explained by Taxi trips and average speed \(R² = 0\.603\), while Taxi average duration retains more unexplained variation \(R² = 0\.300; residual IQR = 1\.086\)\. The same pattern is stronger for FHVHV: average distance is moderately explained by trips and speed \(R² = 0\.547\), whereas average duration remains largely distinct \(R² = 0\.196; residual IQR = 1\.023\)\. Subway Ridership explains only 15\.2% of the variation in Subway Transfers, confirming that Transfers preserves a separate transfer\-hub signal despite its smaller matched sample\. Overall, duration appears more additive than distance for both for\-hire modes, while Subway Transfers provides the clearest distinct signal beyond its baseline counterpart\.


### Compare Top\-Zone Overlap

Correlation is useful, but zone selection is also about which places a metric actually highlights\. This check compares the top candidate zones with the strongest baseline zones, allowing for either same\-direction or opposite\-tail overlap\.

In [8]:
# ---------------------------------------------------------------------
# Compare top-zone overlap between supplemental and baseline metrics
# ---------------------------------------------------------------------

# Define the share of Taxi Zones treated as the "top" or "bottom" tail
# within each congestion-pricing period. Using 20% keeps the comparison
# focused on the zones with the strongest metric values.
TOP_ZONE_FRACTION = 0.20

# Collect one summary record for every supplemental-versus-comparison pair.
top_zone_overlap_records = []

for candidate_metric, comparison_metrics in (
    SUPPLEMENTAL_COMPARISON_METRICS.items()
):
    for comparison_metric in comparison_metrics:

        # Store the overlap result separately for the pre- and post-CP periods
        # before averaging them into one candidate-level summary.
        period_overlap_records = []

        for period_label, period_df in (
            zone_period_representation_df
            .groupby(
                "pre_post_cp",
                observed=True,
            )
        ):
            # Restrict the comparison to Taxi Zone-period rows where both
            # measures are observed.
            metric_df = (
                period_df[
                    [
                        "taxi_zone_id",
                        candidate_metric,
                        comparison_metric,
                    ]
                ]
                .dropna()
            )

            # Skip comparisons with too few matched Taxi Zones to support
            # a meaningful tail-overlap calculation.
            if len(metric_df) < 10:
                continue

            # Select the top 20% of matched Taxi Zones, with a minimum of one
            # zone in very small samples.
            top_n = max(
                1,
                int(
                    np.ceil(
                        metric_df[
                            "taxi_zone_id"
                        ].nunique()
                        * TOP_ZONE_FRACTION
                    )
                ),
            )

            # Identify the Taxi Zones with the highest candidate values,
            # highest comparison values, and lowest comparison values.
            candidate_top_zones = set(
                metric_df
                .nlargest(
                    top_n,
                    candidate_metric,
                )[
                    "taxi_zone_id"
                ]
            )

            comparison_top_zones = set(
                metric_df
                .nlargest(
                    top_n,
                    comparison_metric,
                )[
                    "taxi_zone_id"
                ]
            )

            comparison_bottom_zones = set(
                metric_df
                .nsmallest(
                    top_n,
                    comparison_metric,
                )[
                    "taxi_zone_id"
                ]
            )

            # Measure same-direction overlap and inverse overlap separately.
            # A high candidate may align with either a high or low baseline
            # value, depending on the direction of the relationship.
            same_tail_overlap = (
                len(
                    candidate_top_zones
                    & comparison_top_zones
                )
                / top_n
            )

            opposite_tail_overlap = (
                len(
                    candidate_top_zones
                    & comparison_bottom_zones
                )
                / top_n
            )

            # Keep the stronger alignment because an inverse relationship can
            # still indicate substantial redundancy between two measures.
            best_overlap = max(
                same_tail_overlap,
                opposite_tail_overlap,
            )

            best_alignment = (
                "same_high_tail"
                if same_tail_overlap
                >= opposite_tail_overlap
                else "opposite_tail"
            )

            period_overlap_records.append(
                {
                    "period": period_label,
                    "best_overlap": best_overlap,
                    "best_alignment": best_alignment,
                }
            )

        # Convert the pre/post period results into a temporary review table.
        period_overlap_df = pd.DataFrame(
            period_overlap_records
        )

        # Summarize the average strongest tail overlap across available periods
        # and record whether the relationship most often aligned in the same
        # direction or in opposite tails.
        top_zone_overlap_records.append(
            {
                "candidate_metric": candidate_metric,
                "candidate_label": METRIC_LABELS[
                    candidate_metric
                ],
                "comparison_metric": comparison_metric,
                "comparison_label": METRIC_LABELS[
                    comparison_metric
                ],
                "mean_best_tail_overlap_pct": (
                    round(
                        period_overlap_df[
                            "best_overlap"
                        ].mean()
                        * 100,
                        1,
                    )
                    if not period_overlap_df.empty
                    else np.nan
                ),
                "period_count": len(
                    period_overlap_df
                ),
                "dominant_alignment": (
                    period_overlap_df[
                        "best_alignment"
                    ]
                    .mode()
                    .iloc[0]
                    if not period_overlap_df.empty
                    else np.nan
                ),
            }
        )

# Assemble the final comparison table and sort each supplemental candidate's
# baseline comparisons from strongest to weakest tail overlap.
supplemental_top_zone_overlap_df = (
    pd.DataFrame(
        top_zone_overlap_records
    )
    .sort_values(
        [
            "candidate_metric",
            "mean_best_tail_overlap_pct",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    supplemental_top_zone_overlap_df
)

,candidate_metric,candidate_label,comparison_metric,comparison_label,mean_best_tail_overlap_pct,period_count,dominant_alignment
0,fhvhv_avg_trip_distance,FHVHV average distance,fhvhv_avg_trip_speed,FHVHV average speed,61.3,2,same_high_tail
1,fhvhv_avg_trip_distance,FHVHV average distance,fhvhv_avg_trip_duration,FHVHV average duration,53.8,2,same_high_tail
2,fhvhv_avg_trip_distance,FHVHV average distance,fhvhv_trip_count,FHVHV trips,51.9,2,opposite_tail
3,fhvhv_avg_trip_duration,FHVHV average duration,fhvhv_avg_trip_distance,FHVHV average distance,53.8,2,same_high_tail
4,fhvhv_avg_trip_duration,FHVHV average duration,fhvhv_trip_count,FHVHV trips,34.0,2,same_high_tail
5,fhvhv_avg_trip_duration,FHVHV average duration,fhvhv_avg_trip_speed,FHVHV average speed,18.9,2,same_high_tail
6,subway_transfers,Subway transfers,subway_ridership,Subway ridership,53.9,2,same_high_tail
7,taxi_avg_trip_distance,Taxi average distance,taxi_avg_trip_speed,Taxi average speed,69.8,2,same_high_tail
8,taxi_avg_trip_distance,Taxi average distance,taxi_avg_trip_duration,Taxi average duration,61.3,2,same_high_tail
9,taxi_avg_trip_distance,Taxi average distance,taxi_trip_count,Taxi trips,57.5,2,opposite_tail


Findings\. The tail\-overlap analysis reinforces the distinction between the supplemental candidates\. Taxi average distance shows substantial overlap with Taxi average speed \(69\.8%\), Taxi average duration \(61\.3%\), and the opposite tail of Taxi trip count \(57\.5%\), indicating that it largely identifies zones already differentiated by the baseline\. Taxi average duration overlaps less with trip count and speed \(33\.0% each\), suggesting a more distinct signal despite its relationship with distance\. FHVHV average distance similarly overlaps strongly with speed \(61\.3%\) and duration \(53\.8%\), while FHVHV average duration shows much weaker overlap with speed \(18\.9%\) and trip count \(34\.0%\)\. Subway Transfers overlaps with the highest\-ridership zones only 53\.9% of the time, confirming that it partly tracks system scale but also identifies a distinct set of transfer\-oriented Taxi Zones\. Overall, duration remains the stronger supplemental candidate for both for\-hire modes, while Subway Transfers continues to show meaningful nonredundant structure\.

### Check Pre/Post Stability

A clustering feature should describe durable zone structure, not just a one\-period quirk\. This cell checks whether supplemental metrics rank zones similarly before and after congestion pricing\.

In [9]:
# ---------------------------------------------------------------------
# Check pre/post stability of supplemental zone rankings
# ---------------------------------------------------------------------

# Collect one stability summary for each supplemental candidate metric.
supplemental_stability_records = []

for candidate_metric in SUPPLEMENTAL_CANDIDATE_METRICS:

    # Reshape the Taxi Zone-period values so each Taxi Zone has separate
    # pre-CP and post-CP columns for the current supplemental metric.
    # Retain only zones observed in both periods so the comparison is paired.
    metric_wide_df = (
        zone_period_representation_df[
            [
                "taxi_zone_id",
                "pre_post_cp",
                candidate_metric,
            ]
        ]
        .dropna(
            subset=[candidate_metric]
        )
        .pivot(
            index="taxi_zone_id",
            columns="pre_post_cp",
            values=candidate_metric,
        )
        .dropna(
            subset=[
                "pre_cp",
                "post_cp",
            ]
        )
    )

    # Require a minimally adequate matched sample and variation in both periods
    # before calculating a rank correlation.
    has_variation = (
        len(metric_wide_df) >= 10
        and metric_wide_df[
            "pre_cp"
        ].nunique() > 1
        and metric_wide_df[
            "post_cp"
        ].nunique() > 1
    )

    # Spearman correlation measures whether Taxi Zones maintain a similar
    # relative ordering from the pre-CP period to the post-CP period.
    pre_post_rank_correlation = (
        metric_wide_df[
            "pre_cp"
        ].corr(
            metric_wide_df[
                "post_cp"
            ],
            method="spearman",
        )
        if has_variation
        else np.nan
    )

    # Define the top quintile separately within each period using the same
    # 20% threshold applied in the prior tail-overlap analysis.
    top_n = (
        max(
            1,
            int(
                np.ceil(
                    len(metric_wide_df)
                    * TOP_ZONE_FRACTION
                )
            ),
        )
        if len(metric_wide_df)
        else 0
    )

    # Identify the highest-valued Taxi Zones before and after congestion pricing.
    pre_top_zones = (
        set(
            metric_wide_df
            .nlargest(
                top_n,
                "pre_cp",
            )
            .index
        )
        if top_n
        else set()
    )

    post_top_zones = (
        set(
            metric_wide_df
            .nlargest(
                top_n,
                "post_cp",
            )
            .index
        )
        if top_n
        else set()
    )

    # Measure the share of pre-CP top-quintile zones that remain in the
    # post-CP top quintile. This complements the full rank correlation by
    # focusing on stability among the most extreme zones.
    top_quintile_retention_pct = (
        len(
            pre_top_zones
            & post_top_zones
        )
        / top_n
        * 100
        if top_n
        else np.nan
    )

    # Store both the broad rank-stability measure and the top-tail retention
    # measure for the current supplemental candidate.
    supplemental_stability_records.append(
        {
            "candidate_metric": candidate_metric,
            "candidate_label": METRIC_LABELS[
                candidate_metric
            ],
            "matched_taxi_zones": len(
                metric_wide_df
            ),
            "pre_post_rank_correlation": (
                round(
                    pre_post_rank_correlation,
                    3,
                )
                if pd.notna(
                    pre_post_rank_correlation
                )
                else np.nan
            ),
            "top_quintile_retention_pct": (
                round(
                    top_quintile_retention_pct,
                    1,
                )
                if pd.notna(
                    top_quintile_retention_pct
                )
                else np.nan
            ),
        }
    )

# Assemble the candidate-level stability review table.
supplemental_pre_post_stability_df = pd.DataFrame(
    supplemental_stability_records
)

display(
    supplemental_pre_post_stability_df
)

,candidate_metric,candidate_label,matched_taxi_zones,pre_post_rank_correlation,top_quintile_retention_pct
0,taxi_avg_trip_distance,Taxi average distance,262,0.955,79.2
1,taxi_avg_trip_duration,Taxi average duration,262,0.917,69.8
2,fhvhv_avg_trip_distance,FHVHV average distance,262,0.971,88.7
3,fhvhv_avg_trip_duration,FHVHV average duration,262,0.967,88.7
4,subway_transfers,Subway transfers,153,0.930,87.1


Findings\. All five supplemental measures show strong pre/post stability in Taxi Zone rankings, with Spearman correlations ranging from 0\.917 to 0\.971\. FHVHV distance and duration are the most stable, each retaining 88\.7% of pre\-CP top\-quintile zones in the post\-CP period, while Subway Transfers is similarly stable despite being observed for fewer zones \(ρ = 0\.930; 87\.1% retention\)\. Taxi distance and duration are also highly persistent, though duration shows somewhat more movement among the highest\-ranked zones\. Overall, these measures appear to reflect durable Taxi Zone characteristics rather than transient period\-specific noise, so stability does not argue against retaining any of them\.

### Check Subway Transfer\-Hub Signal

Finally, we treat subway transfers as its own question\. A transfer metric may be worth keeping if it identifies transfer\-heavy zones that ridership alone does not fully explain, especially within zones where subway activity is actually present\.

In [10]:
# ---------------------------------------------------------------------
# Check subway transfer-hub signal
# ---------------------------------------------------------------------

# Isolate the Subway measures and interpretation fields needed to evaluate
# whether Transfers captures a distinct interchange-hub role beyond Ridership.
subway_zone_period_df = zone_period_representation_df[
    [
        "taxi_zone_id",
        "zone",
        "borough",
        "pre_post_cp",
        "subway_ridership",
        "subway_transfers",
    ]
].copy()

# Express transfer activity relative to total ridership so zones of different
# overall Subway scale can be compared on transfer intensity rather than volume.
# Only compute the ratio where Ridership is positive and therefore provides
# a meaningful denominator.
subway_zone_period_df["transfers_per_100_riders"] = np.where(
    subway_zone_period_df["subway_ridership"] > 0,
    (
        subway_zone_period_df["subway_transfers"]
        / subway_zone_period_df["subway_ridership"]
    )
    * 100,
    np.nan,
)

# Collapse the pre- and post-CP observations to one interpretive summary per
# Taxi Zone. Averaging across periods highlights the persistent transfer-hub
# role of each zone rather than a single-period fluctuation.
subway_transfer_hub_signal_df = (
    subway_zone_period_df
    .dropna(
        subset=["transfers_per_100_riders"]
    )
    .groupby(
        [
            "taxi_zone_id",
            "zone",
            "borough",
        ],
        observed=True,
    )
    .agg(
        avg_subway_ridership=(
            "subway_ridership",
            "mean",
        ),
        avg_subway_transfers=(
            "subway_transfers",
            "mean",
        ),
        avg_transfers_per_100_riders=(
            "transfers_per_100_riders",
            "mean",
        ),
        period_count=(
            "pre_post_cp",
            "nunique",
        ),
    )
    .reset_index()
)

# Retain only zones with positive average Ridership, then rank the strongest
# transfer-intensity zones. The top 15 provide a compact review of whether
# the metric surfaces plausible interchange hubs rather than merely the
# highest-volume Subway zones.
subway_transfer_hub_signal_df = (
    subway_transfer_hub_signal_df.loc[
        subway_transfer_hub_signal_df[
            "avg_subway_ridership"
        ]
        > 0
    ]
    .sort_values(
        "avg_transfers_per_100_riders",
        ascending=False,
    )
    .head(15)
    .reset_index(drop=True)
)

# Round displayed values for readability while preserving the underlying
# interpretation of scale, transfer volume, and transfer intensity.
for numeric_column in [
    "avg_subway_ridership",
    "avg_subway_transfers",
    "avg_transfers_per_100_riders",
]:
    subway_transfer_hub_signal_df[
        numeric_column
    ] = subway_transfer_hub_signal_df[
        numeric_column
    ].round(3)

display(
    subway_transfer_hub_signal_df
)

,taxi_zone_id,zone,borough,avg_subway_ridership,avg_subway_transfers,avg_transfers_per_100_riders,period_count
0,92,Flushing,Queens,8013.951,2334.827,29.082,2
1,206,Saint George/New Brighton,Staten Island,1088.602,293.744,26.974,2
2,93,Flushing Meadows-Corona Park,Queens,3957.501,1006.296,25.353,2
3,130,Jamaica,Queens,10994.439,2565.396,23.257,2
4,193,Queensbridge/Ravenswood,Queens,1215.851,239.130,19.418,2
5,183,Pelham Bay,Bronx,1353.108,220.002,16.238,2
6,28,Briarwood/Jamaica Hills,Queens,884.409,142.669,16.118,2
7,242,Van Nest/Morris Park,Bronx,481.237,77.794,16.051,2
8,149,Madison,Brooklyn,3263.225,523.385,16.016,2
9,220,Spuyten Duyvil/Kingsbridge,Bronx,2357.751,350.208,14.845,2


Findings\. Transfer intensity identifies a distinct set of Subway\-oriented Taxi Zones rather than simply reproducing the highest\-ridership locations\. Flushing leads with 29\.1 transfers per 100 riders, followed by Saint George/New Brighton at 27\.0 and Flushing Meadows–Corona Park at 25\.4, while Jamaica combines both high ridership and high transfer intensity\. Several lower\-volume zones, including Queensbridge/Ravenswood, Pelham Bay, Briarwood/Jamaica Hills, and Van Nest/Morris Park, also rank highly, showing that transfer activity is not determined by total ridership alone\. Because all listed zones are observed in both policy periods, the pattern appears persistent and supports retaining Subway Transfers as an interpretable transfer\-hub signal beyond Subway Ridership\.

> ⚖️ Decision\. We will keep the clustering representation simple and carry forward the full 10\-metric set: the seven baseline mobility measures plus taxi\_avg\_trip\_duration, fhvhv\_avg\_trip\_duration, and subway\_transfers\. The duration metrics add useful trip\-composition signal beyond trip count and speed, and subway\_transfers captures transfer\-hub structure that ridership alone does not fully explain\.

We will not add Taxi/FHVHV average distance to the primary clustering representation\. Distance remains available as a sensitivity option later, but for the main matrix it adds more overlap than clarity\.

## 1\.4\.3\.2 Period\-Level Summary and Temporal Shape

This section decides how the 10 retained primary metrics should become clustering inputs at the \`Taxi Zone × pre\_post\_cp\` grain\. We start by checking whether mean and median tell materially different stories, then move to the bigger representation choice: raw temporal\-bucket features versus a cleaner split between overall level and temporal shape\.

The goal is to keep the eventual 3\.1\.1 matrix interpretable\. Raw bucket features are easy to build but mix volume and timing together; level plus normalized temporal shape separates how much activity a zone has from when that activity happens\. This section compares those options and ends with the representation rule that 3\.1\.1 should implement\.

### Period\-Level Summary Statistic

First we compare mean and median summaries for the retained 10 metrics\. If they are close, mean is the simpler period\-level statistic because it preserves total exposure across the full policy period; if they diverge sharply, median may better represent typical zone behavior\.

In [11]:
# ---------------------------------------------------------------------
# Compare mean and median summaries for primary clustering metrics
# ---------------------------------------------------------------------

mean_median_comparison_records = []

for metric in PRIMARY_CLUSTERING_METRICS:
    metric_values = mobility_panel_df[
        [
            "taxi_zone_id",
            "pre_post_cp",
            metric,
        ]
    ].dropna(subset=[metric])

    zone_period_summary_df = (
        metric_values
        .groupby(
            GRAIN_COLUMNS,
            observed=True,
        )[metric]
        .agg(
            period_mean="mean",
            period_median="median",
        )
        .reset_index()
    )

    # Compare mean and median in relative terms while avoiding division by zero.
    zone_period_summary_df["mean_median_pct_gap"] = np.where(
        zone_period_summary_df["period_mean"].abs() > 0,
        (
            (
                zone_period_summary_df["period_mean"]
                - zone_period_summary_df["period_median"]
            ).abs()
            / zone_period_summary_df["period_mean"].abs()
        )
        * 100,
        np.nan,
    )

    mean_median_comparison_records.append(
        {
            "metric": metric,
            "metric_label": METRIC_LABELS[metric],
            "zone_period_count": len(zone_period_summary_df),
            "median_pct_gap": round(
                zone_period_summary_df["mean_median_pct_gap"].median(),
                3,
            ),
            "p90_pct_gap": round(
                zone_period_summary_df["mean_median_pct_gap"].quantile(0.90),
                3,
            ),
            "max_pct_gap": round(
                zone_period_summary_df["mean_median_pct_gap"].max(),
                3,
            ),
        }
    )

mean_median_comparison_df = pd.DataFrame(
    mean_median_comparison_records
)

display(mean_median_comparison_df)

,metric,metric_label,zone_period_count,median_pct_gap,p90_pct_gap,max_pct_gap
0,taxi_trip_count,Taxi trips,526,22.755,100.000,100.000
1,taxi_avg_trip_speed,Taxi average speed,525,5.660,10.509,27.921
2,fhvhv_trip_count,FHVHV trips,526,3.031,15.208,100.000
3,fhvhv_avg_trip_speed,FHVHV average speed,525,3.489,6.404,13.992
4,subway_ridership,Subway ridership,309,19.020,32.596,90.102
5,bus_trip_count,Bus trips,504,4.695,15.564,47.614
6,avg_bus_speed,Average bus speed,504,3.929,8.249,16.520
7,taxi_avg_trip_duration,Taxi average duration,525,8.839,20.420,78.570
8,fhvhv_avg_trip_duration,FHVHV average duration,525,1.242,3.611,54.923
9,subway_transfers,Subway transfers,309,26.879,45.329,100.000


Findings\. This comparison is mainly telling us that one period\-level summary is not enough by itself\. Speed and duration metrics have fairly small mean/median gaps, so their period averages behave like stable summaries\. Count\-like metrics are different: Taxi trips, Subway ridership, and especially Subway transfers show much larger gaps, meaning activity is unevenly distributed across days and buckets\. That does not make the mean wrong; it means the mean is useful for overall exposure, but we also need a separate temporal\-shape representation to capture when that activity happens\.

### Temporal\-Profile Encoding

Next we compare two temporal encodings\. Raw bucket features keep each metric\-bucket value directly, while level plus shape separates each metric's overall zone\-period level from its normalized within\-period bucket profile\.

In [12]:
# ---------------------------------------------------------------------
# Build raw temporal-bucket representation preview
# ---------------------------------------------------------------------

raw_bucket_summary_df = (
    mobility_panel_df
    .groupby(
        TEMPORAL_GRAIN_COLUMNS,
        observed=True,
    )[PRIMARY_CLUSTERING_METRICS]
    .mean()
    .reset_index()
)

raw_bucket_feature_count = (
    len(PRIMARY_CLUSTERING_METRICS)
    * len(TEMPORAL_BUCKET_ORDER)
)

raw_bucket_representation_summary_df = pd.DataFrame(
    {
        "metric": [
            "zone_period_bucket_rows",
            "taxi_zone_count",
            "policy_period_count",
            "temporal_bucket_count",
            "primary_metric_count",
            "raw_bucket_feature_count",
        ],
        "value": [
            len(raw_bucket_summary_df),
            raw_bucket_summary_df["taxi_zone_id"].nunique(),
            raw_bucket_summary_df["pre_post_cp"].nunique(),
            raw_bucket_summary_df["temporal_bucket"].nunique(),
            len(PRIMARY_CLUSTERING_METRICS),
            raw_bucket_feature_count,
        ],
    }
)

display(raw_bucket_representation_summary_df)

,metric,value
0,zone_period_bucket_rows,5260
1,taxi_zone_count,263
2,policy_period_count,2
3,temporal_bucket_count,10
4,primary_metric_count,10
5,raw_bucket_feature_count,100


Findings\. The raw temporal\-bucket option has the expected shape: 263 Taxi Zones, two policy periods, and ten temporal buckets, for 5,260 zone\-period\-bucket rows\. With 10 retained metrics, this would produce 100 raw metric\-by\-bucket features\. That is manageable, but it also means each feature mixes two things at once: how large the zone\-period signal is and when that signal is concentrated\.

In [13]:
# ---------------------------------------------------------------------
# Build level plus normalized temporal-shape representation preview
# ---------------------------------------------------------------------

level_shape_records = []

for metric in PRIMARY_CLUSTERING_METRICS:
    metric_bucket_df = raw_bucket_summary_df[
        [
            "taxi_zone_id",
            "pre_post_cp",
            "temporal_bucket",
            metric,
        ]
    ].copy()

    metric_level_df = (
        metric_bucket_df
        .groupby(
            GRAIN_COLUMNS,
            observed=True,
        )[metric]
        .mean()
        .reset_index(name="period_level")
    )

    metric_bucket_df = metric_bucket_df.merge(
        metric_level_df,
        on=GRAIN_COLUMNS,
        how="left",
    )

    # Shape is a within-zone-period index: 100 means the bucket equals that
    # metric's own period-level average for the same Taxi Zone and period.
    metric_bucket_df["shape_index"] = np.where(
        metric_bucket_df["period_level"].abs() > 0,
        metric_bucket_df[metric] / metric_bucket_df["period_level"] * 100,
        np.nan,
    )

    level_shape_records.append(
        {
            "metric": metric,
            "metric_label": METRIC_LABELS[metric],
            "level_feature_count": 1,
            "shape_feature_count": metric_bucket_df["temporal_bucket"].nunique(),
            "median_shape_index": round(metric_bucket_df["shape_index"].median(), 3),
            "p10_shape_index": round(metric_bucket_df["shape_index"].quantile(0.10), 3),
            "p90_shape_index": round(metric_bucket_df["shape_index"].quantile(0.90), 3),
        }
    )

level_shape_representation_preview_df = pd.DataFrame(
    level_shape_records
)

level_shape_feature_count = int(
    level_shape_representation_preview_df["level_feature_count"].sum()
    + level_shape_representation_preview_df["shape_feature_count"].sum()
)

display(level_shape_representation_preview_df)

,metric,metric_label,level_feature_count,shape_feature_count,median_shape_index,p10_shape_index,p90_shape_index
0,taxi_trip_count,Taxi trips,1,10,88.889,35.351,175.374
1,taxi_avg_trip_speed,Taxi average speed,1,10,97.091,71.402,132.459
2,fhvhv_trip_count,FHVHV trips,1,10,100.450,48.069,150.006
3,fhvhv_avg_trip_speed,FHVHV average speed,1,10,95.486,78.624,125.601
4,subway_ridership,Subway ridership,1,10,76.889,18.912,203.777
5,bus_trip_count,Bus trips,1,10,86.949,40.006,168.353
6,avg_bus_speed,Average bus speed,1,10,97.942,81.827,124.034
7,taxi_avg_trip_duration,Taxi average duration,1,10,98.042,71.752,130.945
8,fhvhv_avg_trip_duration,FHVHV average duration,1,10,99.690,88.250,112.570
9,subway_transfers,Subway transfers,1,10,68.106,12.716,218.652


Findings\. The level\-plus\-shape preview shows that the retained metrics have very different temporal profiles\. Demand and volume measures, especially Subway ridership, Subway transfers, Taxi trips, and Bus trips, swing much more across temporal buckets, while speed and duration measures stay closer to their zone\-period averages\. This is exactly why the separated representation is useful: level features capture how much activity or performance a zone has overall, while shape features capture the timing pattern without forcing those two ideas into the same raw bucket value\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>